# Hash Map (separate chaining, from scratch)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Concurrency, Hash Tables, OOP & Design Patterns · **Difficulty/Frequency:** Rare (2/10)

> **See also:** [`25. Hash_Table`](../25.%20Hash_Table/25.%20Hash_Table.ipynb) is a *different* question despite the similar name — it adds **TTL expiration** on top of a hash table. This notebook covers the hash table itself; that one covers what happens when entries expire.

## Concepts

**What this problem is really testing:**
- What makes a hash map **O(1)**, and precisely when that claim stops being true
- **Collision resolution**, and why a resize is not optional
- Whether you can say something real about **thread safety** beyond "add a lock"

**First-principles primer — what is each piece?**

- **The core trick.** An array gives O(1) access *by index*. A hash function turns a key into an index. Chain the two and you get O(1) access *by key*. That is the entire idea — everything else is repairing the places it breaks.
- **Collision.** Infinitely many keys, finitely many slots, so two keys **must** eventually share an index (pigeonhole). Not an edge case — a certainty. **Separate chaining** stores a linked list at each slot; **open addressing** probes for another free slot.
- **Load factor** = `size / capacity` = the average chain length. At 0.75 the average chain is under one node, so a lookup is "hash, then look at roughly one thing". Let it reach 10 and every lookup walks ten nodes: still technically O(1) *amortised in the size of the table*, but ten times slower in practice.
- **Amortised O(1).** Resizing is a genuine O(n) operation. But because capacity **doubles**, resizes happen at 16, 32, 64, … inserts, so the total rehash work across n inserts is `n + n/2 + n/4 + ... < 2n` — O(1) per insert on average. Growing by a *constant* instead of doubling would make it O(n) per insert, which is the classic dynamic-array argument.

**"O(1)" deserves an asterisk, and interviewers listen for it:**

| Case | Cost | When |
|---|---|---|
| Average | **O(1)** | a good hash spreading keys evenly |
| Worst | **O(n)** | every key lands in one bucket |

The worst case is not hypothetical. Feeding a server keys chosen to collide is a real denial-of-service attack (**hash flooding**), and it is why Python randomises string hashing per process by default.

**Why `put` must handle the update case.** If the key already exists you overwrite the value and return **without** incrementing `size`. Miss that and `size` counts insertions rather than entries, the load factor drifts upward, and the map resizes for no reason.

**Simple worked example.** Capacity 4, three keys that collide:

```
hash("a") % 4 = 1        buckets[1] -> ("a",1)
hash("b") % 4 = 1        buckets[1] -> ("b",2) -> ("a",1)      <- PREPENDED, O(1)
hash("c") % 4 = 3        buckets[3] -> ("c",3)
```

`get("a")`: hash to bucket 1, walk the chain — `"b"`? no. `"a"`? yes → 1. Two comparisons, not three keys scanned.

Prepending rather than appending is deliberate: adding to the head is O(1), while appending would walk to the tail and make insertion O(chain length).

## Problem Statement

Implement a hash map with `put(key, value)` and `get(key)`.

- What is the time complexity?
- How would you make it thread-safe?

```python
m = MyHashMap()
m.put("a", 1)
m.put("a", 2)      # update, not insert
m.get("a")         # -> 2
m.get("missing")   # -> not found
```

### Approach 1 — Naive (a list of pairs)

**Idea:** keep `(key, value)` pairs in a list; scan it for every operation.

Correct, and the baseline that shows what the hash buys you. Every operation walks the whole structure, so it is O(n) — the exact cost a hash map exists to remove.

**Time complexity:** **O(n)** for both `put` and `get`.

**Space complexity:** O(n).

In [ ]:
from typing import Any, List, Optional, Tuple

_MISSING = object()          # a sentinel no user value can ever equal


class NaiveMap:
    """Baseline: a list of pairs, scanned linearly."""

    def __init__(self) -> None:
        self.pairs: List[List[Any]] = []

    def put(self, key: Any, value: Any) -> None:
        for pair in self.pairs:                  # O(n) scan
            if pair[0] == key:
                pair[1] = value                  # UPDATE: do not grow
                return
        self.pairs.append([key, value])

    def get(self, key: Any, default: Any = _MISSING) -> Any:
        for k, v in self.pairs:                  # O(n) scan
            if k == key:
                return v
        return default

    def __len__(self) -> int:
        return len(self.pairs)

### Approach 2 — Optimal (separate chaining with resize)

**Idea:** an array of buckets; each bucket a linked list of entries that hashed to that index.

**Four details worth defending:**

- **Prepend, do not append.** Inserting at the head is O(1); appending would walk to the tail, making insertion cost the chain length.
- **Check for an existing key *before* inserting.** Otherwise you get two nodes with the same key, `get` returns whichever comes first, and `size` is wrong.
- **A sentinel for "not found", not `-1`.** The official answer returns `-1`, which is indistinguishable from a stored `-1`. A private sentinel object cannot collide with any user value.
- **Rehash directly into the new array.** Calling the public `put` during a resize re-checks the load factor and re-scans for duplicate keys — both pointless, since the entries are known unique and the table was just grown. It also creates a recursion hazard that only *happens* to be safe at a 2× growth factor.

**Time complexity:** **O(1) average** for `put` and `get`; **O(n) worst case** if every key collides. Resize is O(n), amortised to O(1) per insert by doubling.

**Space complexity:** O(n) entries plus O(capacity) for the bucket array.

In [ ]:
class Entry:
    __slots__ = ("key", "value", "next")

    def __init__(self, key: Any, value: Any, nxt: "Optional[Entry]" = None) -> None:
        self.key, self.value, self.next = key, value, nxt


class MyHashMap:
    """Separate chaining, with load-factor-driven resizing."""

    def __init__(self, capacity: int = 16, load_factor: float = 0.75) -> None:
        self.capacity = capacity
        self.load_factor = load_factor
        self.size = 0
        self.buckets: List[Optional[Entry]] = [None] * capacity
        self.resizes = 0                         # instrumentation for the tests below

    def _index(self, key: Any) -> int:
        return hash(key) % self.capacity         # hash() may be negative; % makes it a valid index

    def put(self, key: Any, value: Any) -> None:
        idx = self._index(key)
        node = self.buckets[idx]
        while node is not None:
            if node.key == key:
                node.value = value               # UPDATE - crucially, size does NOT change
                return
            node = node.next
        self.buckets[idx] = Entry(key, value, self.buckets[idx])   # PREPEND: O(1)
        self.size += 1
        if self.size / self.capacity > self.load_factor:
            self._resize()

    def get(self, key: Any, default: Any = _MISSING) -> Any:
        node = self.buckets[self._index(key)]
        while node is not None:                  # walk only THIS bucket's chain
            if node.key == key:
                return node.value
            node = node.next
        if default is _MISSING:
            raise KeyError(key)                  # unambiguous: a stored -1 is not "missing"
        return default

    def remove(self, key: Any) -> bool:
        idx = self._index(key)
        node, prev = self.buckets[idx], None
        while node is not None:
            if node.key == key:
                if prev is None:
                    self.buckets[idx] = node.next    # it was the head
                else:
                    prev.next = node.next            # unlink from the middle
                self.size -= 1
                return True
            prev, node = node, node.next
        return False

    def _resize(self) -> None:
        old = self.buckets
        self.capacity *= 2                       # DOUBLE, so rehashing amortises to O(1)/insert
        self.buckets = [None] * self.capacity
        self.resizes += 1
        for head in old:
            node = head
            while node is not None:
                nxt = node.next                  # save it: we are about to overwrite node.next
                i = self._index(node.key)
                node.next = self.buckets[i]      # rehash DIRECTLY - no load-factor re-check,
                self.buckets[i] = node           # no duplicate scan, and the nodes are reused
                node = nxt

    def __contains__(self, key: Any) -> bool:
        return self.get(key, None) is not None or self._has(key)

    def _has(self, key: Any) -> bool:
        node = self.buckets[self._index(key)]
        while node is not None:
            if node.key == key:
                return True
            node = node.next
        return False

    def __len__(self) -> int:
        return self.size

    def items(self):
        for head in self.buckets:
            node = head
            while node is not None:
                yield node.key, node.value
                node = node.next

    def chain_lengths(self) -> List[int]:
        out = []
        for head in self.buckets:
            n, node = 0, head
            while node is not None:
                n, node = n + 1, node.next
            out.append(n)
        return out

### Approach 3 — Thread safety, and what it actually costs

**Idea:** the question asks how to make it thread-safe, and there are three answers with genuinely different trade-offs.

- **One global lock** (`SynchronizedHashMap`). Correct, trivial, and it serialises **everything** — including reads that could safely have run in parallel. Fine when contention is low.
- **A readers–writer lock.** Many concurrent readers, exclusive writers. A real win here, because — unlike the [LRU cache](../10.%20LRU_Cache/10.%20LRU_Cache.ipynb), where `get` mutates the recency list — a hash map's `get` is a **genuine read**. Worth saying explicitly: it is the property that makes the optimisation legal.
- **Lock striping** (`StripedHashMap`), which is what Java's `ConcurrentHashMap` did. Keep N independent locks and guard bucket `i` with lock `i % N`. Two threads touching different buckets never contend, so throughput scales with N. The cost: **any operation spanning all buckets** — `size()`, `resize()`, iteration — must acquire *every* lock, in a fixed order to avoid deadlock. That is why `ConcurrentHashMap.size()` is famously approximate.

**The general lesson:** finer locks buy concurrency and pay in complexity, and the bill arrives at whichever operation needs to see the whole structure at once.

**Time complexity:** unchanged; the difference is contention, not asymptotics.

**Space complexity:** O(n) plus O(stripes).

In [ ]:
import threading


class SynchronizedHashMap(MyHashMap):
    """One lock for everything. Correct, and serialises reads that need not be."""

    def __init__(self, capacity: int = 16, load_factor: float = 0.75) -> None:
        super().__init__(capacity, load_factor)
        self._lock = threading.Lock()

    def put(self, key: Any, value: Any) -> None:
        with self._lock:
            super().put(key, value)

    def get(self, key: Any, default: Any = _MISSING) -> Any:
        with self._lock:
            return super().get(key, default)

    def remove(self, key: Any) -> bool:
        with self._lock:
            return super().remove(key)


class StripedHashMap:
    """Lock striping: N independent shards, so unrelated keys never contend."""

    def __init__(self, stripes: int = 16, capacity: int = 16) -> None:
        self.stripes = stripes
        self._maps = [MyHashMap(capacity) for _ in range(stripes)]
        self._locks = [threading.Lock() for _ in range(stripes)]

    def _shard(self, key: Any) -> int:
        return hash(key) % self.stripes

    def put(self, key: Any, value: Any) -> None:
        i = self._shard(key)
        with self._locks[i]:                     # only ONE stripe is blocked
            self._maps[i].put(key, value)

    def get(self, key: Any, default: Any = _MISSING) -> Any:
        i = self._shard(key)
        with self._locks[i]:
            return self._maps[i].get(key, default)

    def __len__(self) -> int:
        # A whole-structure operation needs EVERY lock - always in the same order,
        # or two such calls could deadlock against each other.
        for lk in self._locks:
            lk.acquire()
        try:
            return sum(len(m) for m in self._maps)
        finally:
            for lk in reversed(self._locks):
                lk.release()

## Verification

The basics, then the cases that separate a working hash map from a plausible one: updates that must not grow the map, deliberate collisions, resizing correctness, the `-1` sentinel trap, and real threads.

In [ ]:
import random
from concurrent.futures import ThreadPoolExecutor

# --- Basics ---
m = MyHashMap()
m.put("a", 1)
m.put("b", 2)
assert m.get("a") == 1 and m.get("b") == 2
assert len(m) == 2

# --- put on an existing key UPDATES, and must not grow the map ---
m.put("a", 99)
assert m.get("a") == 99
assert len(m) == 2, "an update must not increase size - or the load factor drifts"

# --- THE sentinel trap: -1 is a legitimate value ---
m.put("neg", -1)
assert m.get("neg") == -1
assert m.get("absent", default=None) is None
try:
    m.get("absent")
except KeyError:
    pass
else:
    raise AssertionError("a missing key with no default must raise, not return a magic value")
# A stored None is present, and distinguishable from absent
m.put("none", None)
assert m.get("none") is None and m._has("none")
assert not m._has("absent")

# --- remove ---
assert m.remove("a") is True
assert len(m) == 3
assert m.remove("a") is False, "removing twice is False, not an error"
try:
    m.get("a")
except KeyError:
    pass
else:
    raise AssertionError("a removed key must be gone")

# --- Deliberate collisions: everything must still work ---
class Collide:
    """Every instance hashes to 0, forcing one long chain."""

    def __init__(self, n): self.n = n
    def __hash__(self): return 0
    def __eq__(self, other): return isinstance(other, Collide) and self.n == other.n
    def __repr__(self): return f"Collide({self.n})"


c = MyHashMap(capacity=8, load_factor=100.0)      # never resize, so the chain really grows
keys = [Collide(i) for i in range(20)]
for i, k in enumerate(keys):
    c.put(k, i * 10)
assert len(c) == 20
for i, k in enumerate(keys):
    assert c.get(k) == i * 10, f"{k} lost in the chain"
lengths = c.chain_lengths()
assert max(lengths) == 20 and sum(lengths) == 20, (
    f"all 20 keys must be in ONE bucket: {lengths}"
)
# Removing from the middle of a long chain must relink correctly
assert c.remove(keys[10]) is True
assert c.get(keys[9]) == 90 and c.get(keys[11]) == 110, "the chain must survive an unlink"
assert len(c) == 19

# --- Resizing ---
r = MyHashMap(capacity=4, load_factor=0.75)
assert r.capacity == 4
for i in range(3):
    r.put(f"k{i}", i)
assert r.capacity == 4 and r.resizes == 0, (
    "3/4 == 0.75 exactly, and the check is STRICTLY greater - so no resize yet"
)
r.put("k3", 3)
assert r.capacity == 8 and r.resizes == 1, "4/4 = 1.0 > 0.75 triggers exactly one resize"
for i in range(4):
    assert r.get(f"k{i}") == i, "every entry must survive a rehash"

# Resizing must not lose, duplicate or corrupt anything
big = MyHashMap(capacity=2)
expected = {}
for i in range(500):
    big.put(f"key{i}", i)
    expected[f"key{i}"] = i
assert len(big) == 500
assert big.capacity >= 500 / 0.75, "the table must have grown"
assert big.resizes >= 8
for k, v in expected.items():
    assert big.get(k) == v
assert dict(big.items()) == expected, "iteration must see exactly the inserted pairs"

# The load factor is genuinely maintained
assert len(big) / big.capacity <= 0.75 + 1e-9

# With a decent hash, chains stay short - which IS the O(1) claim
lengths = big.chain_lengths()
assert max(lengths) <= 6, f"a healthy table should have short chains, saw {max(lengths)}"

# --- Agreement with the naive map, and with dict, on randomised operations ---
random.seed(107)
for _ in range(200):
    mine, naive, ref = MyHashMap(capacity=2), NaiveMap(), {}
    for _ in range(random.randint(0, 120)):
        k = f"k{random.randrange(15)}"
        r = random.random()
        if r < 0.6:
            v = random.randrange(100)
            mine.put(k, v); naive.put(k, v); ref[k] = v
        elif r < 0.85:
            assert mine.get(k, None) == naive.get(k, None) == ref.get(k)
        else:
            mine.remove(k); ref.pop(k, None)
            naive.pairs = [p for p in naive.pairs if p[0] != k]
    assert len(mine) == len(ref) == len(naive)
    assert dict(mine.items()) == ref

# --- Keys of mixed types, and the negative-hash case ---
mixed = MyHashMap()
for k in ("str", 42, -42, 3.14, (1, 2), True, None):
    mixed.put(k, repr(k))
for k in ("str", 42, -42, 3.14, (1, 2), True, None):
    assert mixed.get(k) == repr(k), f"{k!r} must round-trip"
assert mixed.get(-42) == "-42", "a negative hash must still map to a valid index"

# --- Thread safety ---
sync = SynchronizedHashMap()


def hammer(worker):
    for i in range(200):
        sync.put(f"w{worker}-{i}", i)
        sync.get(f"w{worker}-{i}", None)


with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(hammer, range(8)))
assert len(sync) == 8 * 200, f"concurrent puts lost entries: {len(sync)}"
for w in range(8):
    for i in range(200):
        assert sync.get(f"w{w}-{i}") == i

striped = StripedHashMap(stripes=8)


def hammer_striped(worker):
    for i in range(200):
        striped.put(f"w{worker}-{i}", i)
        striped.get(f"w{worker}-{i}", None)


with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(hammer_striped, range(8)))
assert len(striped) == 8 * 200, "striping must not lose entries either"
for w in range(8):
    assert striped.get(f"w{w}-0") == 0

# The UNSYNCHRONISED map is expected to be unsafe - documented, not asserted,
# since a race may or may not manifest on any given run.
print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Open addressing instead of chaining.** All entries live in the array itself; a collision probes for the next free slot. It wins on **cache locality** — probing walks contiguous memory rather than chasing pointers — which in practice often beats chaining outright. The price is that **deletion becomes hard**: you cannot simply empty a slot, because that would break the probe chain for any key that hopped over it. You need a **tombstone** marking "occupied but deleted", and tombstones accumulate until you rehash. It is also far more sensitive to load factor, degrading sharply past ~0.7 through clustering — which is why CPython's own dict, which uses open addressing, resizes at 2/3 rather than 3/4.
- **`remove`.** Implemented above. The detail worth naming is the `prev` pointer: unlinking from the middle needs the *previous* node, which is why a singly linked chain requires tracking it as you walk. Shrinking on a low load factor is possible but usually skipped — it invites **thrashing**, where a workload hovering at the boundary resizes back and forth. The standard fix is hysteresis: grow at 0.75, shrink only below 0.25.
- **Adversarial keys.** The worst case is not theoretical. If an attacker can choose keys — HTTP parameter names, JSON fields — they can force every key into one bucket and turn every request into an O(n) scan. That is **hash flooding**, and it took down web frameworks in 2011. Defences: a **randomised per-process seed** (Python's default since 3.3), a keyed hash like SipHash, or converting a long chain into a balanced tree, which is what Java 8's `HashMap` does above 8 nodes per bin, capping the worst case at O(log n).
- **How `ConcurrentHashMap` beats a synchronised map.** Three things, and the third is the interesting one: **lock striping** so unrelated keys never contend; **volatile reads** so lookups need no lock at all; and **CAS** for lock-free insertion into an empty bin. The consequence is that whole-structure operations lose their exactness — `size()` is approximate, and iteration is *weakly consistent*: it never throws, but it may or may not reflect concurrent modifications. That is a deliberate trade, not a bug.
- **Efficient iteration.** Walking the bucket array is O(capacity), not O(n) — with a sparse table you spend most of the time skipping empty slots. Threading a doubly linked list through the entries makes iteration O(n) *and* gives a defined order. That structure is exactly [`12. Linked_Hash_Map`](../12.%20Linked_Hash_Map/12.%20Linked_Hash_Map.ipynb), which is this problem's natural sequel.
- **Why doubling, specifically.** Growing by a constant (say +16) makes the total rehash work across n inserts O(n²). Doubling makes it `n + n/2 + n/4 + ... < 2n`, so amortised O(1) per insert. It is the same argument that makes a dynamic array's `append` amortised O(1) — worth naming, because it is the reason "resize is O(n)" does not ruin the complexity claim.

## Empirical complexity check

Two things worth measuring.

**First**, the point of the whole exercise: the naive list scan is O(n) per operation, the hash map O(1).

**Second**, and more instructive, what happens when the hash is **adversarial** — every key colliding into one bucket. That is the O(n) worst case, and it turns the map back into the very linked list it was built to replace.

| Growth when the entry count doubles | What it means |
|---|---|
| ~2x | linear — a scan, or a single degenerate chain |
| ~1x | constant — hash, then look at roughly one node |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

LOOKUPS = 2000


def make_keys(n):
    return (n,)


def run_naive(n):
    m = NaiveMap()
    for i in range(n):
        m.put(f"k{i}", i)
    for i in range(LOOKUPS):
        m.get(f"k{i % n}", None)                 # O(n) per lookup


def run_hashmap(n):
    m = MyHashMap()
    for i in range(n):
        m.put(f"k{i}", i)
    for i in range(LOOKUPS):
        m.get(f"k{i % n}", None)                 # O(1) average


def run_adversarial(n):
    m = MyHashMap(load_factor=1e9)               # never resize
    keys = [Collide(i) for i in range(n)]
    for i, k in enumerate(keys):
        m.put(k, i)
    for i in range(LOOKUPS):
        m.get(keys[i % n], None)                 # every key in ONE bucket: O(n) per lookup


benchmark(
    {"Approach 1 - list scan O(n)": run_naive,
     "Approach 2 - hash map O(1) average": run_hashmap,
     "Approach 2 - hash map, ADVERSARIAL keys O(n)": run_adversarial},
    make_keys,
    sizes=[250, 500, 1000, 2000],
    repeats=1,
)

## Patterns learned

- **A hash map is an array plus a function from keys to indices.** Arrays are O(1) by index; the hash supplies the index. Everything else in the implementation exists to repair where that breaks.
- **Collisions are certain, not exceptional.** Finitely many slots, unboundedly many keys. Chaining and open addressing are the two repairs, with different failure modes.
- **"O(1)" needs its asterisk.** O(1) *average, with a good hash*; O(n) worst case. Saying only the first half is the red flag; naming hash flooding is what shows you mean it.
- **Doubling is what makes resizing free.** `n + n/2 + n/4 + ... < 2n` amortises to O(1) per insert. Growing by a constant would be O(n) per insert — the same argument as a dynamic array's `append`.
- **The load factor *is* the average chain length.** 0.75 keeps lookups at "roughly one node". It is a dial between memory and speed, not a magic constant.
- **Never use an in-band value for "not found".** `-1` is a perfectly good thing to store. Use a sentinel, a `(found, value)` pair, or raise — the same lesson as [Deep Key Search](../7.%20Deep_Key_Search_Nested_JSON/7.%20Deep_Key_Search_Nested_JSON.ipynb).
- **Do not re-enter your public API from internal maintenance.** Rehashing through `put` re-checks the load factor and re-scans for duplicates that cannot exist. Insert directly into the new array.
- **Finer locks buy concurrency and bill you at the whole-structure operations.** Striping makes per-key work parallel and makes `size()` expensive or approximate. That is the trade `ConcurrentHashMap` makes deliberately.